In [1]:
#required imports
from sklearn.metrics import roc_auc_score
import glob, os, math, sys,json,time
import numpy as np, pandas as pd
from pathlib import Path

In [2]:
#declare gene-drug map
single_drugs = {
    "rifampicin" : ["rpoB"],
    "pyrazinamide": ["pncA"],
    "capreomycin" : ["tlyA"],
    "amikacin"    : ["eis"]
}

multi_drugs = {
    "streptomycin": ["rpsL", "gid"],
    "isoniazid"   : ["katG", "inhA"],
    "ethionamide" : ["ethA", "ethR","inhA"],
    "ethambutol"  : ["embC","embA","embB"],
    "moxifloxacin": ["gyrA", "gyrB"],
    "levofloxacin": ["gyrA", "gyrB"]
}

all_drugs = {**single_drugs, **multi_drugs}   # merge dicts

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 1: convert to float16 memory mapping and meta files for efficient memory

In [4]:
# from tqdm.auto import tqdm  
import tqdm
# ──────────────────────────────────────────────────────────────
# 0.  one‑time conversion: NPZ  →  float16 mem‑map + meta
# ──────────────────────────────────────────────────────────────
#     *.npz files that hold token tensors  →  lightweight
#     float-16 memory-mapped arrays + a tiny “meta” file.
#
#     Why ?
#       • fp16 cuts RAM / disk roughly in half.
#       • mmap lets PyTorch stream slices straight off disk.
# ──────────────────────────────────────────────────────────────


# -------------------------------------------------------------------
# 1) only the exceptions live here
#    (key = (gene, drug)  →  sub-folder to prepend)
# -------------------------------------------------------------------
SPECIAL_DIRS = {
    ("gyrA", "levofloxacin") : "data/latest/embeddings/gyrA_LEV",
    ("gyrB", "levofloxacin") : "data/latest/embeddings/gyrB_LEV",
    ("gyrA", "moxifloxacin") : "data/latest/embeddings/gyrA_MOX",
    ("gyrB", "moxifloxacin") : "data/latest/embeddings/gyrB_MOX",
    ("ethA", "ethionamide")  : "data/latest/embeddings/ethA_ETH",
    ("ethR", "ethionamide")  : "data/latest/embeddings/ethR_ETH",
    ("inhA", "ethionamide")  : "data/latest/embeddings/inhA_ETH",
    
}

# -------------------------------------------------------------------
# 2) one helper – return the directory that actually holds the .npz
# -------------------------------------------------------------------
def embeddings_root(gene: str, drug: str | None = None) -> Path:
    return Path(SPECIAL_DIRS.get((gene, drug),
                                 f"data/latest/embeddings/{gene}"))


In [5]:
def npz_to_memmap_mean(gene: str, *, drug: str | None = None):
    root = embeddings_root(gene, drug)
    src_glob = root / "mean" / f"{gene}_mean_chunk_*.npz"

    for npz in tqdm.tqdm(sorted(glob.glob(str(src_glob))), desc=f"convert mean {gene}"):
        meta_out = npz.replace(".npz", "_meta.npz")
        mmap_out = npz.replace(".npz", ".mmap")
        if os.path.exists(meta_out):  # skip if done
            continue

        z = np.load(npz)
        if "mean_embedding" not in z:
            print(f"  ⤷ skip {os.path.basename(npz)} (no 'mean_embedding')")
            continue

        ids = z["identifier"].astype(str)
        mean = z["mean_embedding"].astype("float16")  # shape = (N, 320)
        mm = np.memmap(mmap_out, mode="w+", dtype="float16", shape=mean.shape)
        mm[:] = mean
        mm.flush()

        np.savez_compressed(meta_out,
                            identifier=ids,
                            shape=mean.shape,
                            mmap_path=os.path.basename(mmap_out))
    print(f"{gene}: mean conversion finished")


In [ ]:
import time

special_drugs = {"levofloxacin", "moxifloxacin","ethionamide"}   # the ones with MOX / LEV folders

for drug, genes in all_drugs.items():
    print(f"\n===== {drug.upper()} =====")

    for g in genes:
        t0 = time.perf_counter()

        # -----------------------------------------------------------------
        # pass drug=drug only if this drug has the special folder structure
        # -----------------------------------------------------------------
        if drug in special_drugs:                 # safe, boolean expression
            npz_to_memmap_mean(gene=g, drug=drug)      # keyword arguments ≠ positional
        else:
            npz_to_memmap_mean(gene=g, drug=drug)      # keyword arguments ≠ positional

        dt = time.perf_counter() - t0
        print(f"  {g}: conversion finished in {dt/60:.2f} min")


## Step 3: multi gene data processing

In [6]:
from tqdm.auto import tqdm

def build_label_map(genes,drug):
    """
    genes  : ["gyrA", "gyrB"]
    returns: { isolate_id -> 0/1 }  (0 = S, 1 = R)
    """
    label_map = {}

    for g in genes:
        csv = Path(f"data/latest/sequence_data_csv/{g}_{drug.upper()}_combined_sequence_data.csv")
        df  = pd.read_csv(csv, usecols=["Filename", "Phenotype"])
        df["id"]    = df["Filename"].astype(str)
        df["label"] = (df["Phenotype"] == "R").astype(int)
        # if an isolate appears in both files, assert the labels agree
        for _id, lbl in zip(df["id"], df["label"]):
            if _id in label_map and label_map[_id] != lbl:
                raise ValueError(f"Phenotype disagreement for {_id} between genes")
            label_map[_id] = lbl

    return label_map

In [7]:
class SeqMeanMultiGeneDataset(torch.utils.data.Dataset):
    """
    Multi-gene mean embeddings (concat of 320-dim per gene).
    """
    def __init__(self, genes, drug, label_map, keep_ids=None):
        self.genes = genes
        self.drug  = drug
        self.label = label_map
        self.blocks = {}

        for g in genes:
            data_path = embeddings_root(g, drug)
            metas = sorted((data_path / "mean").glob("*_meta.npz"))
            gene_blocks = []
            for meta_file in metas:
                meta = np.load(meta_file, allow_pickle=True)
                mm_path = str(meta_file).replace("_meta.npz", ".mmap")
                mm = np.memmap(mm_path, mode="r", dtype="float16", shape=tuple(meta["shape"]))
                ids = meta["identifier"].astype(str)
                gene_blocks.append((ids, mm))
            self.blocks[g] = gene_blocks

        # intersect ids across all genes
        id_sets = [set(id_ for ids, _ in self.blocks[g] for id_ in ids) for g in genes]
        common_ids = set.intersection(*id_sets)

        # filter
        ids = common_ids & set(label_map.keys())
        if keep_ids is not None:
            ids &= set(keep_ids)
        self.ids = sorted(ids)

    def __len__(self):  
        return len(self.ids)

    def _row(self, gene, ident):
        for ids, mm in self.blocks[gene]:
            w = np.where(ids == ident)[0]
            if w.size:
                return torch.from_numpy(mm[w[0]].astype("float32"))  # (320,)
        raise KeyError(f"{ident} missing in {gene}")

    def __getitem__(self, idx):
        ident = self.ids[idx]
        parts = [self._row(g, ident) for g in self.genes]
        x = torch.cat(parts)  # (320*num_genes,)
        y = torch.tensor(self.label[ident], dtype=torch.float32)
        return x, y


class SeqMeanMemmapMap(torch.utils.data.Dataset):
    """
    Single-gene mean embeddings.
    Each item → (320,) float32, label
    """
    def __init__(self, meta_paths, label_dict, keep_ids=None):
        self.blocks = []
        self.lookup = []
        self.label  = label_dict

        for bidx, p in enumerate(meta_paths):
            m = np.load(p, allow_pickle=True)
            ids = m["identifier"].astype(str)
            shape = tuple(m["shape"])
            mmap_path = str(p).replace("_meta.npz", ".mmap")
            mm = np.memmap(mmap_path, dtype="float16", mode="r", shape=shape)
            self.blocks.append((ids, mm))

            for r, id_ in enumerate(ids):
                if id_ in label_dict and (keep_ids is None or id_ in keep_ids):
                    self.lookup.append((bidx, r))

    def __len__(self):  
        return len(self.lookup)

    def __getitem__(self, idx):
        b, r = self.lookup[idx]
        ids, mm = self.blocks[b]
        ident = ids[r]
        x = torch.from_numpy(mm[r].astype("float32"))
        y = torch.tensor(self.label[ident], dtype=torch.float32)
        return x, y


In [8]:
from sklearn.model_selection import train_test_split

def build_train_test_split(drug, test_size=0.2, seed=42):
    """Load sequence CSV(s) for a drug, split into train/test, return filenames + labels."""

    if drug in single_drugs:
        gene = single_drugs[drug][0]
        paths = [f"data/latest/sequence_data_csv/{gene}_{drug.upper()}_combined_sequence_data.csv"]
    else:
        genes = multi_drugs[drug]
        paths = [f"data/latest/sequence_data_csv/{g}_{drug.upper()}_combined_sequence_data.csv"
                 for g in genes]

    dfs = []
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Expected file not found: {p}")
        dfs.append(pd.read_csv(p, usecols=["Filename", "Phenotype"]))

    ph = pd.concat(dfs, ignore_index=True)

    files  = ph["Filename"].astype(str).values
    labels = (ph["Phenotype"] == "R").astype(int).tolist()   # ← cast to list of ints

    X_train, X_test, y_train, y_test = train_test_split(
        files, labels, test_size=test_size, random_state=seed, stratify=labels
    )

    # Force Python int
    y_train = [int(y) for y in y_train]
    y_test  = [int(y) for y in y_test]

    return (X_train, y_train), (X_test, y_test)

## Step 4: MLP Model

In [9]:
import torch.nn as nn, torch.nn.functional as F, math
class MeanMLP(nn.Module):
    """simple MLP for fixed-length  (in_dim,) → logit"""
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

## step 5: Per-token and PCA runs& SHAP residue

In [10]:
# ──────────────────────────────────────────────────────────────
#  Train token model (full 320‑dim or PCA‑K)  +  logging
# ──────────────────────────────────────────────────────────────
# High-level flow
#   1) Build the right Dataset  (single-gene ⟹ TokenMemmapMap
#                                multi-gene ⟹ MultiGeneConcat / PCA-variant)
#   2) Split → DataLoaders
#   3) Instantiate ProteinCNN1x1
#   4) Train w/ pos_weight + bias-freeze trick
#   5) Log AUC & ACC each epoch, save model + CSV history
# --------------------------------------------------------------
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, random_split


In [11]:
def evaluate_auc(model, dataset, device="cuda"):
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    preds, gold = [], []
    model.eval()
    with torch.no_grad():
        for xb,yb in loader:
            xb = xb.to(device)
            preds.extend(torch.sigmoid(model(xb)).cpu().numpy())
            gold.extend(yb.numpy())
    if len(np.unique(gold)) < 2:
        return 0.5
    return roc_auc_score(gold, preds)


In [12]:
def train_token_seqmean(
    gene,
    drug,
    train_files, train_labels,
    test_files, test_labels,
    in_dim=320,
    batch_size=32,
    n_epochs=20,
    lr=5e-4,
    freeze_bias_frac=0.25,
    out_root="data/latest/results",
):
    t0 = time.time()
    run_dir = f"{out_root}/prediction/esm/{drug}_seqmean{in_dim}"
    os.makedirs(run_dir, exist_ok=True)

    # ───────────────────────────────
    # Label maps
    # ───────────────────────────────
    train_label_map = dict(zip(train_files, train_labels))
    test_label_map  = dict(zip(test_files,  test_labels))
    label_map = {**train_label_map, **test_label_map}  # merged

    # ───────────────────────────────
    # Dataset construction
    # ───────────────────────────────
    if drug in multi_drugs:  # multi-gene drug
        genes = multi_drugs[drug]
        tr_ds = SeqMeanMultiGeneDataset(genes, drug, label_map, keep_ids=train_files)
        te_ds = SeqMeanMultiGeneDataset(genes, drug, label_map, keep_ids=test_files)
        in_features = 320 * len(genes)
    else:  # single gene drug
        gene = all_drugs[drug][0]
        meta_paths = sorted(glob.glob(f"{embeddings_root(gene, drug)}/mean/*_meta.npz"))
        tr_ds = SeqMeanMemmapMap(meta_paths, label_map, keep_ids=train_files)
        te_ds = SeqMeanMemmapMap(meta_paths, label_map, keep_ids=test_files)
        in_features = 320

    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    # ───────────────────────────────
    # Model
    # ───────────────────────────────
    model = MeanMLP(in_dim=in_features).to("cuda" if torch.cuda.is_available() else "cpu")

    # pos weight
    labels_arr = np.fromiter(train_label_map.values(), dtype=np.int32)
    nR = labels_arr.sum()
    nS = len(labels_arr) - nR
    pos_weight = torch.tensor(nS / (nR + 1e-8), dtype=torch.float32).to(model.net[0].weight.device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # bias init
    freeze_ep = max(1, int(n_epochs * freeze_bias_frac))
    with torch.no_grad():
        pR = nR / (nR + nS + 1e-8)
        model.net[-1].bias.fill_(math.log(pR / (1 - pR)))
    model.net[-1].bias.requires_grad = False

    # ───────────────────────────────
    # Training
    # ───────────────────────────────
    hist = []
    device = model.net[0].weight.device
    for ep in range(1, n_epochs + 1):
        model.train()
        for xb, yb in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        if ep == freeze_ep:
            model.net[-1].bias.requires_grad = True

        # test evaluation
        model.eval()
        probs, ys = [], []
        with torch.no_grad():
            for xb, yb in te_ld:
                xb = xb.to(device)
                probs.append(torch.sigmoid(model(xb)).cpu())
                ys.append(yb)
        prob = torch.cat(probs).numpy()
        yt = torch.cat(ys).numpy()
        auc = roc_auc_score(yt, prob) if len(np.unique(yt)) > 1 else 0.5
        acc = ((prob > 0.5) == yt).mean()
        print(f"ep {ep:02d}/{n_epochs}  test_auc={auc:.3f}  test_acc={acc:.3f}")
        hist.append({"epoch": ep, "test_auc": float(auc), "test_acc": float(acc)})
    # ───────────────────────────────
    # Final AUCs on train and test
    # ───────────────────────────────
    train_auc = evaluate_auc(model, tr_ds, device=device)
    test_auc  = evaluate_auc(model, te_ds, device=device)

    results = pd.DataFrame({
        "Drug": [drug],
        "Train_AUC": [train_auc],
        "Test_AUC": [test_auc],
    })
    results.to_csv(f"{run_dir}/{drug}_train_test_auc.csv", index=False)
    print(f"{drug}: Train AUC={train_auc:.3f} | Test AUC={test_auc:.3f}")
    # ───────────────────────────────
    # Save
    # ───────────────────────────────
    torch.save(model.state_dict(), f"{run_dir}/{drug}_model.pt")
    pd.DataFrame(hist).to_csv(f"{run_dir}/{drug}_auc_history.csv", index=False)

    print(f"Done — saved to {run_dir} in {(time.time() - t0)/60:.1f} min")
    return model, tr_ds, te_ds, pd.DataFrame(hist)


In [13]:
# ------------------------------------------------------------------
# 0. Which experiments do you want to run?
# ------------------------------------------------------------------
TODO = [
    ("pyrazinamide", 1, "seqmean"),
    ("rifampicin",   1, "seqmean"),
    ("amikacin",     1, "seqmean"),
    ("capreomycin",  1, "seqmean"), 
    ("moxifloxacin", 1, "seqmean"),
    ("levofloxacin", 1, "seqmean"),
    ("ethambutol",   1, "seqmean"),
    ("isoniazid",    1, "seqmean"),
    ("ethionamide",  1, "seqmean"),
    ("streptomycin", 1, "seqmean"),
]

# ------------------------------------------------------------------
# 1. Loop and train/test
# ------------------------------------------------------------------
results = {}
for drug, in_dim, tag in TODO:
    print(f"\n=== {drug}  |  {tag}  ===============================")

    # build train/test split
    (train_files, y_train), (test_files, y_test) = build_train_test_split(drug)

    # train model
    model, tr_ds, te_ds, hist = train_token_seqmean(
        gene=None,
        drug=drug,
        train_files=train_files,
        train_labels=y_train,
        test_files=test_files,
        test_labels=y_test,
        in_dim=in_dim,
        batch_size=32,
        n_epochs=20,
        lr=5e-4,
    )

    results[(drug, in_dim)] = {"model": model, "hist": hist}



=== pyrazinamide  |  seqmean  ===============================


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ep 01/20  test_auc=0.659  test_acc=0.165
ep 02/20  test_auc=0.676  test_acc=0.857
ep 03/20  test_auc=0.672  test_acc=0.865
ep 04/20  test_auc=0.674  test_acc=0.896
ep 05/20  test_auc=0.685  test_acc=0.865
ep 06/20  test_auc=0.685  test_acc=0.867
ep 07/20  test_auc=0.685  test_acc=0.867
ep 08/20  test_auc=0.690  test_acc=0.896
ep 09/20  test_auc=0.692  test_acc=0.876
ep 10/20  test_auc=0.694  test_acc=0.876
ep 11/20  test_auc=0.698  test_acc=0.162
ep 12/20  test_auc=0.699  test_acc=0.874
ep 13/20  test_auc=0.699  test_acc=0.874
ep 14/20  test_auc=0.699  test_acc=0.899
ep 15/20  test_auc=0.703  test_acc=0.895
ep 16/20  test_auc=0.705  test_acc=0.874
ep 17/20  test_auc=0.708  test_acc=0.889
ep 18/20  test_auc=0.712  test_acc=0.901
ep 19/20  test_auc=0.714  test_acc=0.890
ep 20/20  test_auc=0.714  test_acc=0.884
pyrazinamide: Train AUC=0.693 | Test AUC=0.714
Done — saved to data/latest/results/prediction/esm/pyrazinamide_seqmean1 in 0.4 min

=== rifampicin  |  seqmean  ====================